# DATATHON 2026 — Revenue & COGS Forecasting Pipeline

**Objective:** Train daily Revenue and COGS forecasting models (2023-01-01 → 2024-07-01)

**Pipeline Overview:**
1. Config & Imports
2. Load Data
3. Data Validation
4. Feature Engineering (aggregate → calendar → merge → lag/rolling)
5. Train / Validation / Test Split + Seasonal Features
6. Hyperparameter Optimization (Optuna)
7. Train Optimized Models (LightGBM + XGBoost)
8. Ensemble Weight Optimization
9. Generate Final Submission
10. Recursive Forecast Pipeline (advanced auto-regressive strategy)

## 1. Configuration and Imports

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import pandas as pd
from pathlib import Path

# Set random seeds for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Path configuration
BASE_PATH = Path('../data')
RAW_PATH = BASE_PATH / 'raw'
OUTPUT_PATH = BASE_PATH / 'output'
MODEL_PATH = Path('../models')
MODEL_PATH.mkdir(exist_ok=True)

# Date split configuration
TRAIN_START = '2012-07-04'
TRAIN_END = '2021-12-31'
VAL_START = '2022-01-01'
VAL_END = '2022-12-31'
TEST_START = '2023-01-01'
TEST_END = '2024-07-01'

print(f"Random Seed: {RANDOM_SEED}")
print(f"Data Path: {BASE_PATH}")
print(f"Train Period: {TRAIN_START} to {TRAIN_END}")
print(f"Validation Period: {VAL_START} to {VAL_END}")
print(f"Test Period: {TEST_START} to {TEST_END}")


In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-darkgrid')

print("All imports successful!")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"lightgbm: {lgb.__version__}")
print(f"xgboost: {xgb.__version__}")


# ============================================================================
# UTILITY: EVALUATION METRICS
# ============================================================================

def calculate_metrics(y_true, y_pred, model_name="Model"):
    """Compute MAE / RMSE / MAPE / R² and print a summary."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    mask = y_true != 0
    mape = (np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]).mean() * 100
            if mask.sum() > 0 else np.inf)

    print(f"\n{model_name} Metrics:")
    print(f"  MAE:  {mae:,.2f}")
    print(f"  RMSE: {rmse:,.2f}")
    print(f"  MAPE: {mape:.2f}%")
    print(f"  R²:   {r2:.4f}")

    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}


## 2. Load Data

In [ ]:
# ============================================================================
# LOAD DATA
# ============================================================================

# Load target data (daily revenue + COGS)
print("Loading sales.csv...")
sales_df = pd.read_csv(RAW_PATH / 'sales.csv', parse_dates=['Date'])
sales_df.rename(columns={'Date': 'date'}, inplace=True)
print(f"Sales shape: {sales_df.shape}")
print(f"Date range: {sales_df['date'].min()} to {sales_df['date'].max()}")
print(f"Columns: {sales_df.columns.tolist()}")

# Load master table (order_item level)
print("\nLoading master_table.csv...")
master_df = pd.read_csv(OUTPUT_PATH / 'master_table.csv')
print(f"Master table shape: {master_df.shape}")
print(f"Columns: {master_df.columns.tolist()[:10]}... (first 10)")

# Parse date columns in master_table
date_cols = ['order_date', 'payment_date', 'ship_date', 'delivery_date', 'return_date', 'review_date']
for col in date_cols:
    if col in master_df.columns:
        master_df[col] = pd.to_datetime(master_df[col], errors='coerce')

print(f"\nMaster table date range (order_date): {master_df['order_date'].min()} to {master_df['order_date'].max()}")
print(f"\nFirst few rows of sales:")
print(sales_df.head())


## 3. Data Validation and Exploration

In [ ]:
# ============================================================================
# DATA VALIDATION
# ============================================================================

print("=" * 80)
print("SALES DATA VALIDATION")
print("=" * 80)

# Check for missing dates
date_range = pd.date_range(start=sales_df['date'].min(), end=sales_df['date'].max(), freq='D')
missing_dates = set(date_range) - set(sales_df['date'])
print(f"\nTotal dates in range: {len(date_range)}")
print(f"Dates in sales: {len(sales_df)}")
print(f"Missing dates: {len(missing_dates)}")

# Check for nulls
print(f"\nNull values in sales:")
print(sales_df.isnull().sum())

# Check for duplicates
print(f"\nDuplicate dates in sales: {sales_df['date'].duplicated().sum()}")

# Summary statistics
print(f"\nSales summary statistics:")
print(sales_df.describe())

print("\n" + "=" * 80)
print("MASTER TABLE VALIDATION")
print("=" * 80)

# Check master table data types
print(f"\nMaster table data types:")
print(master_df.dtypes)

# Check null percentages
print(f"\nNull percentages in master table:")
null_pct = (master_df.isnull().sum() / len(master_df) * 100).sort_values(ascending=False)
print(null_pct[null_pct > 0].head(20))

# Check for negative values in revenue/cogs
if 'line_revenue' in master_df.columns:
    print(f"\nNegative line_revenue: {(master_df['line_revenue'] < 0).sum()}")
if 'line_cogs' in master_df.columns:
    print(f"Negative line_cogs: {(master_df['line_cogs'] < 0).sum()}")

## 4. Feature Engineering

### 4.1 Aggregate Master Table to Daily Level

In [ ]:
# ============================================================================
# AGGREGATE MASTER TABLE TO DAILY LEVEL
# ============================================================================

print("Aggregating master_table to daily level...")

# Filter master table to historical data only (before test period)
master_historical = master_df[master_df['order_date'] < TEST_START].copy()
print(f"Historical master table: {len(master_historical)} rows")
print(f"Date range: {master_historical['order_date'].min()} to {master_historical['order_date'].max()}")

# Aggregate numerical features
daily_features = master_historical.groupby('order_date').agg({
    'order_item_id': 'count',  # Number of order items
    'order_id': 'nunique',     # Number of unique orders
    'customer_id': 'nunique',  # Number of unique customers
    'product_id': 'nunique',   # Number of unique products
    'quantity': ['sum', 'mean', 'std'],
    'unit_price': ['mean', 'std', 'min', 'max'],
    'discount_amount': ['sum', 'mean', 'std'],
    'line_revenue': ['sum', 'mean', 'std'],
    'line_cogs': ['sum', 'mean', 'std'],
}).reset_index()

# Flatten column names
daily_features.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in daily_features.columns.values]
daily_features.rename(columns={'order_date': 'date'}, inplace=True)

print(f"\nDaily features shape: {daily_features.shape}")
print(f"Columns: {daily_features.columns.tolist()}")

# Aggregate categorical features
print("\nAggregating categorical features...")

# Category distribution
if 'category' in master_historical.columns:
    category_daily = master_historical.groupby(['order_date', 'category']).size().unstack(fill_value=0)
    category_daily.columns = [f'category_{col}_count' for col in category_daily.columns]
    category_daily = category_daily.reset_index()
    category_daily.rename(columns={'order_date': 'date'}, inplace=True)
    daily_features = daily_features.merge(category_daily, on='date', how='left')

# Segment distribution
if 'segment' in master_historical.columns:
    segment_daily = master_historical.groupby(['order_date', 'segment']).size().unstack(fill_value=0)
    segment_daily.columns = [f'segment_{col}_count' for col in segment_daily.columns]
    segment_daily = segment_daily.reset_index()
    segment_daily.rename(columns={'order_date': 'date'}, inplace=True)
    daily_features = daily_features.merge(segment_daily, on='date', how='left')

# Region distribution
if 'region' in master_historical.columns:
    region_daily = master_historical.groupby(['order_date', 'region']).size().unstack(fill_value=0)
    region_daily.columns = [f'region_{col}_count' for col in region_daily.columns]
    region_daily = region_daily.reset_index()
    region_daily.rename(columns={'order_date': 'date'}, inplace=True)
    daily_features = daily_features.merge(region_daily, on='date', how='left')

# Payment method distribution
if 'payment_method' in master_historical.columns:
    payment_daily = master_historical.groupby(['order_date', 'payment_method']).size().unstack(fill_value=0)
    payment_daily.columns = [f'payment_{col}_count' for col in payment_daily.columns]
    payment_daily = payment_daily.reset_index()
    payment_daily.rename(columns={'order_date': 'date'}, inplace=True)
    daily_features = daily_features.merge(payment_daily, on='date', how='left')

# Order source distribution
if 'order_source' in master_historical.columns:
    source_daily = master_historical.groupby(['order_date', 'order_source']).size().unstack(fill_value=0)
    source_daily.columns = [f'source_{col}_count' for col in source_daily.columns]
    source_daily = source_daily.reset_index()
    source_daily.rename(columns={'order_date': 'date'}, inplace=True)
    daily_features = daily_features.merge(source_daily, on='date', how='left')

# Calculate ratios and percentages
print("\nCalculating derived features...")

# Promo ratio
if 'promo_id' in master_historical.columns:
    promo_ratio = master_historical.groupby('order_date').apply(
        lambda x: (x['promo_id'].notna() & (x['promo_id'] != 'none')).sum() / len(x)
    ).reset_index()
    promo_ratio.columns = ['date', 'promo_ratio']
    daily_features = daily_features.merge(promo_ratio, on='date', how='left')

# Legacy ratio
if 'is_legacy' in master_historical.columns:
    legacy_ratio = master_historical.groupby('order_date')['is_legacy'].mean().reset_index()
    legacy_ratio.columns = ['date', 'legacy_ratio']
    daily_features = daily_features.merge(legacy_ratio, on='date', how='left')

# Return ratio (if available)
if 'order_status' in master_historical.columns:
    return_ratio = master_historical.groupby('order_date').apply(
        lambda x: (x['order_status'] == 'returned').sum() / len(x)
    ).reset_index()
    return_ratio.columns = ['date', 'return_ratio']
    daily_features = daily_features.merge(return_ratio, on='date', how='left')

# Review ratio
if 'review_id' in master_historical.columns:
    review_ratio = master_historical.groupby('order_date').apply(
        lambda x: (x['review_id'].notna() & (x['review_id'] != 0)).sum() / len(x)
    ).reset_index()
    review_ratio.columns = ['date', 'review_ratio']
    daily_features = daily_features.merge(review_ratio, on='date', how='left')

# Average items per order
daily_features['items_per_order'] = daily_features['order_item_id_count'] / daily_features['order_id_nunique']

# Average revenue per order
daily_features['revenue_per_order'] = daily_features['line_revenue_sum'] / daily_features['order_id_nunique']

# Average revenue per customer
daily_features['revenue_per_customer'] = daily_features['line_revenue_sum'] / daily_features['customer_id_nunique']

# Discount ratio
daily_features['discount_ratio'] = daily_features['discount_amount_sum'] / daily_features['line_revenue_sum']

# Fill any inf or nan from division
daily_features = daily_features.replace([np.inf, -np.inf], np.nan)
daily_features = daily_features.fillna(0)

print(f"\nFinal daily features shape: {daily_features.shape}")
print(f"Columns ({len(daily_features.columns)}): {daily_features.columns.tolist()[:20]}... (first 20)")
print(f"\nFirst few rows:")
print(daily_features.head())

### 4.2 Create Calendar Features

In [ ]:
# ============================================================================
# CALENDAR FEATURES (function definition only — applied to df_all in §4.3)
# ============================================================================

def create_calendar_features(df, date_col='date'):
    """
    Create calendar-based features from date column.
    Safe for time series: all features are derived from the date itself (no leakage).
    Applied once to the full date-range dataframe (df_all) after the merge,
    NOT to daily_features, to avoid duplicate columns during the merge step.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # Basic date parts
    df['year']        = df[date_col].dt.year
    df['month']       = df[date_col].dt.month
    df['day']         = df[date_col].dt.day
    df['day_of_week'] = df[date_col].dt.dayofweek   # 0 = Monday
    df['day_of_year'] = df[date_col].dt.dayofyear
    df['week_of_year']= df[date_col].dt.isocalendar().week.astype(int)
    df['quarter']     = df[date_col].dt.quarter

    # Cyclical encodings
    df['month_sin']        = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']        = np.cos(2 * np.pi * df['month'] / 12)
    df['day_of_week_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_year_sin']  = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_of_year_cos']  = np.cos(2 * np.pi * df['day_of_year'] / 365)
    df['week_of_year_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
    df['week_of_year_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)

    # Boolean indicators
    df['is_weekend']     = (df['day_of_week'] >= 5).astype(int)
    df['is_month_start'] = df[date_col].dt.is_month_start.astype(int)
    df['is_month_end']   = df[date_col].dt.is_month_end.astype(int)
    df['is_quarter_start'] = df[date_col].dt.is_quarter_start.astype(int)
    df['is_quarter_end']   = df[date_col].dt.is_quarter_end.astype(int)
    df['is_year_start']  = df[date_col].dt.is_year_start.astype(int)
    df['is_year_end']    = df[date_col].dt.is_year_end.astype(int)

    # Week of month (1–5)
    df['week_of_month'] = (df['day'] - 1) // 7 + 1

    return df

print("create_calendar_features() defined — will be applied to df_all in §4.3.")


### 4.3 Merge Features with Sales Target

In [ ]:
# ============================================================================
# MERGE FEATURES WITH TARGET
# ============================================================================

print("Merging daily features with sales target...")

# Create complete date range for train + validation + test
all_dates = pd.date_range(start=TRAIN_START, end=TEST_END, freq='D')
df_all = pd.DataFrame({'date': all_dates})

print(f"Complete date range: {len(df_all)} days")

# Merge with sales (target)
df_all = df_all.merge(sales_df[['date', 'Revenue', 'COGS']], on='date', how='left')

# Merge with daily features
df_all = df_all.merge(daily_features, on='date', how='left')

# Add calendar features for all dates (including test)
df_all = create_calendar_features(df_all, 'date')

print(f"\nMerged data shape: {df_all.shape}")
print(f"Columns: {len(df_all.columns)}")

# Check target availability
print(f"\nTarget (Revenue) availability:")
print(f"  Non-null: {df_all['Revenue'].notna().sum()}")
print(f"  Null: {df_all['Revenue'].isna().sum()}")

# Check feature availability
print(f"\nFeature availability (order_item_id_count):")
print(f"  Non-null: {df_all['order_item_id_count'].notna().sum()}")
print(f"  Null: {df_all['order_item_id_count'].isna().sum()}")

# Forward fill features for missing dates (within reason)
# Note: Only fill features that come from master_table, not the target
feature_cols = [col for col in df_all.columns if col not in ['date', 'Revenue', 'COGS']]
df_all[feature_cols] = df_all[feature_cols].fillna(0)

print(f"\nAfter filling nulls:")
print(f"  Null counts: {df_all.isnull().sum().sum()}")

print(f"\nFirst few rows:")
print(df_all.head())

print(f"\nLast few rows:")
print(df_all.tail())

### 4.4 Lag and Rolling Features

In [ ]:
# ============================================================================
# LAG AND ROLLING FEATURES
# ============================================================================

def create_lag_features(df, target_col, lags, date_col='date'):
    """
    Create lag features for a target column.
    IMPORTANT: This function assumes df is sorted by date.
    """
    df = df.copy()
    df = df.sort_values(date_col).reset_index(drop=True)
    
    for lag in lags:
        df[f'{target_col}_lag_{lag}'] = df[target_col].shift(lag)
    
    return df

def create_rolling_features(df, target_col, windows, date_col='date'):
    """
    Create rolling statistics features.
    IMPORTANT: This function assumes df is sorted by date.
    """
    df = df.copy()
    df = df.sort_values(date_col).reset_index(drop=True)
    
    for window in windows:
        # Rolling mean
        df[f'{target_col}_rolling_mean_{window}'] = df[target_col].shift(1).rolling(window=window, min_periods=1).mean()
        
        # Rolling std
        df[f'{target_col}_rolling_std_{window}'] = df[target_col].shift(1).rolling(window=window, min_periods=1).std()
        
        # Rolling min/max
        df[f'{target_col}_rolling_min_{window}'] = df[target_col].shift(1).rolling(window=window, min_periods=1).min()
        df[f'{target_col}_rolling_max_{window}'] = df[target_col].shift(1).rolling(window=window, min_periods=1).max()
    
    return df

def create_expanding_features(df, target_col, date_col='date'):
    """
    Create expanding window statistics.
    """
    df = df.copy()
    df = df.sort_values(date_col).reset_index(drop=True)
    
    df[f'{target_col}_expanding_mean'] = df[target_col].shift(1).expanding(min_periods=1).mean()
    df[f'{target_col}_expanding_std'] = df[target_col].shift(1).expanding(min_periods=1).std()
    
    return df

def create_diff_features(df, target_col, lags, date_col='date'):
    """
    Create difference features.
    """
    df = df.copy()
    df = df.sort_values(date_col).reset_index(drop=True)
    
    for lag in lags:
        df[f'{target_col}_diff_{lag}'] = df[target_col] - df[target_col].shift(lag)
    
    return df

print("Creating lag and rolling features...")
print("This may take a few moments...\n")

# Sort by date first
df_all = df_all.sort_values('date').reset_index(drop=True)

# Define lag periods (in days)
LAG_PERIODS = [1, 7, 14, 28, 56, 365]
ROLLING_WINDOWS = [7, 14, 28, 56, 365]
DIFF_PERIODS = [1, 7, 28, 365]

print(f"Creating lag features for Revenue...")
df_all = create_lag_features(df_all, 'Revenue', LAG_PERIODS, 'date')

print(f"Creating rolling features for Revenue...")
df_all = create_rolling_features(df_all, 'Revenue', ROLLING_WINDOWS, 'date')

print(f"Creating expanding features for Revenue...")
df_all = create_expanding_features(df_all, 'Revenue', 'date')

print(f"Creating difference features for Revenue...")
df_all = create_diff_features(df_all, 'Revenue', DIFF_PERIODS, 'date')

# Also create lag/rolling for key business metrics
print(f"\nCreating lag features for order_id_nunique...")
df_all = create_lag_features(df_all, 'order_id_nunique', [1, 7, 28], 'date')
df_all = create_rolling_features(df_all, 'order_id_nunique', [7, 28], 'date')

print(f"Creating lag features for customer_id_nunique...")
df_all = create_lag_features(df_all, 'customer_id_nunique', [1, 7, 28], 'date')
df_all = create_rolling_features(df_all, 'customer_id_nunique', [7, 28], 'date')

print(f"\nFinal dataset shape: {df_all.shape}")
print(f"Total features: {len(df_all.columns) - 3}  (excluding date, Revenue, COGS)")

# Check for any remaining nulls in features
feature_cols = [col for col in df_all.columns if col not in ['date', 'Revenue', 'COGS']]
null_counts = df_all[feature_cols].isnull().sum()
print(f"\nFeatures with nulls: {(null_counts > 0).sum()}")
if (null_counts > 0).any():
    print("\nTop features with nulls:")
    print(null_counts[null_counts > 0].sort_values(ascending=False).head(10))

## 5. Train/Validation/Test Split

In [ ]:
# ============================================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================================

print("Splitting data by time...")

# Convert split dates to datetime
train_start_dt = pd.to_datetime(TRAIN_START)
train_end_dt = pd.to_datetime(TRAIN_END)
val_start_dt = pd.to_datetime(VAL_START)
val_end_dt = pd.to_datetime(VAL_END)
test_start_dt = pd.to_datetime(TEST_START)
test_end_dt = pd.to_datetime(TEST_END)

# Split datasets
train_df = df_all[(df_all['date'] >= train_start_dt) & (df_all['date'] <= train_end_dt)].copy()
val_df = df_all[(df_all['date'] >= val_start_dt) & (df_all['date'] <= val_end_dt)].copy()
test_df = df_all[(df_all['date'] >= test_start_dt) & (df_all['date'] <= test_end_dt)].copy()

print(f"\nTrain set:")
print(f"  Shape: {train_df.shape}")
print(f"  Date range: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"  Non-null Revenue: {train_df['Revenue'].notna().sum()}")

print(f"\nValidation set:")
print(f"  Shape: {val_df.shape}")
print(f"  Date range: {val_df['date'].min()} to {val_df['date'].max()}")
print(f"  Non-null Revenue: {val_df['Revenue'].notna().sum()}")

print(f"\nTest set:")
print(f"  Shape: {test_df.shape}")
print(f"  Date range: {test_df['date'].min()} to {test_df['date'].max()}")
print(f"  Non-null Revenue: {test_df['Revenue'].notna().sum()}")

# ============================================================================
# WEEKLY SEASONAL MEAN FEATURES (computed from train only — no data leakage)
# These act as strong "anchor" signals so the model learns the yearly weekly pattern.
# ============================================================================
print("\nAdding weekly seasonal mean features...")

week_rev_mean  = train_df.groupby('week_of_year')['Revenue'].mean()
week_cogs_mean = train_df.groupby('week_of_year')['COGS'].mean()
week_rev_std   = train_df.groupby('week_of_year')['Revenue'].std().fillna(0)
week_cogs_std  = train_df.groupby('week_of_year')['COGS'].std().fillna(0)

for split_df in [train_df, val_df, test_df]:
    split_df['revenue_week_seasonal_mean'] = split_df['week_of_year'].map(week_rev_mean)
    split_df['cogs_week_seasonal_mean']    = split_df['week_of_year'].map(week_cogs_mean)
    split_df['revenue_week_seasonal_std']  = split_df['week_of_year'].map(week_rev_std)
    split_df['cogs_week_seasonal_std']     = split_df['week_of_year'].map(week_cogs_std)

print(f"  Added: revenue_week_seasonal_mean, cogs_week_seasonal_mean, "
      f"revenue_week_seasonal_std, cogs_week_seasonal_std")
print(f"  Weeks covered by training: {week_rev_mean.index.nunique()}")

# Feature columns (exclude date and targets; use train_df columns to include new seasonal features)
exclude_cols = ['date', 'Revenue', 'COGS']
feature_cols = [col for col in train_df.columns if col not in exclude_cols]

print(f"\nTotal feature columns: {len(feature_cols)}")
print(f"Week-related features: {[c for c in feature_cols if 'week' in c]}")

# Prepare X and y for train and validation
X_train = train_df[feature_cols].copy()
y_train = train_df['Revenue'].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df['Revenue'].copy()

X_test = test_df[feature_cols].copy()

# Fill any remaining NaN in features with 0
X_train = X_train.fillna(0)
X_val = X_val.fillna(0)
X_test = X_test.fillna(0)

# Replace inf values
X_train = X_train.replace([np.inf, -np.inf], 0)
X_val = X_val.replace([np.inf, -np.inf], 0)
X_test = X_test.replace([np.inf, -np.inf], 0)

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}")

# Verify no nulls or infs
print(f"\nX_train nulls: {X_train.isnull().sum().sum()}")
# Count inf values safely (numeric columns only)
x_train_num = X_train.select_dtypes(include=[np.number]).astype(np.float64)
print(f"X_train infs: {np.isinf(x_train_num.to_numpy()).sum()}")
print(f"y_train nulls: {y_train.isnull().sum()}")


In [ ]:
# ============================================================================
# PREPARE TARGETS FOR REVENUE AND COGS
# ============================================================================

print("=" * 80)
print("PREPARING TARGETS FOR BOTH REVENUE AND COGS")
print("=" * 80)

y_train_revenue = train_df['Revenue'].copy()
y_val_revenue   = val_df['Revenue'].copy()

y_train_cogs = train_df['COGS'].copy()
y_val_cogs   = val_df['COGS'].copy()

print(f"\nRevenue — Train: {y_train_revenue.shape}, Non-null: {y_train_revenue.notna().sum()}")
print(f"Revenue — Val:   {y_val_revenue.shape},   Non-null: {y_val_revenue.notna().sum()}")
print(f"COGS    — Train: {y_train_cogs.shape}, Non-null: {y_train_cogs.notna().sum()}")
print(f"COGS    — Val:   {y_val_cogs.shape},   Non-null: {y_val_cogs.notna().sum()}")

revenue_cogs_ratio = y_train_revenue / y_train_cogs
print(f"\nRevenue/COGS ratio — Mean: {revenue_cogs_ratio.mean():.4f}, "
      f"Median: {revenue_cogs_ratio.median():.4f}, Std: {revenue_cogs_ratio.std():.4f}")


## 6. Hyperparameter Optimization (Optuna)

Search for optimal hyperparameters for LightGBM and XGBoost on the validation set before final training.

In [ ]:
# ============================================================================
# HYPERPARAMETER OPTIMIZATION WITH OPTUNA
# ============================================================================

# Install optuna if not available
try:
    import optuna
    from optuna.samplers import TPESampler
    print("Optuna is available!")
except ImportError:
    print("Installing optuna...")
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"])
    import optuna
    from optuna.samplers import TPESampler
    print("Optuna installed successfully!")

# Suppress optuna logs
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("=" * 80)
print("HYPERPARAMETER OPTIMIZATION")
print("=" * 80)

# Define objective function for LightGBM (Revenue)
def objective_lgb_revenue(trial):
    """Optuna objective function for LightGBM - Revenue."""
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'verbose': -1,
        'seed': RANDOM_SEED,
        'n_jobs': -1
    }
    
    lgb_train_optuna = lgb.Dataset(X_train, label=y_train_revenue)
    lgb_val_optuna = lgb.Dataset(X_val, label=y_val_revenue, reference=lgb_train_optuna)
    
    model = lgb.train(
        params,
        lgb_train_optuna,
        num_boost_round=1000,
        valid_sets=[lgb_val_optuna],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=0)
        ]
    )
    
    y_pred = model.predict(X_val, num_iteration=model.best_iteration)
    rmse = np.sqrt(mean_squared_error(y_val_revenue, y_pred))
    
    return rmse

# Define objective function for XGBoost (Revenue)
def objective_xgb_revenue(trial):
    """Optuna objective function for XGBoost - Revenue."""
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 10.0, log=True),
        'lambda': trial.suggest_float('lambda', 1e-8, 10.0, log=True),
        'seed': RANDOM_SEED,
        'tree_method': 'hist',
        'verbosity': 0
    }
    
    dtrain_optuna = xgb.DMatrix(X_train, label=y_train_revenue)
    dval_optuna = xgb.DMatrix(X_val, label=y_val_revenue)
    
    model = xgb.train(
        params,
        dtrain_optuna,
        num_boost_round=1000,
        evals=[(dval_optuna, 'valid')],
        early_stopping_rounds=50,
        verbose_eval=False
    )
    
    y_pred = model.predict(dval_optuna, iteration_range=(0, model.best_iteration + 1))
    rmse = np.sqrt(mean_squared_error(y_val_revenue, y_pred))
    
    return rmse

# Optimize LightGBM for Revenue
print("\n" + "=" * 80)
print("OPTIMIZING LIGHTGBM HYPERPARAMETERS (REVENUE)")
print("=" * 80)
print("Running 30 trials (this may take 5-10 minutes)...")

study_lgb = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=RANDOM_SEED)
)
study_lgb.optimize(objective_lgb_revenue, n_trials=30, show_progress_bar=True)

print(f"\nBest RMSE: {study_lgb.best_value:,.2f}")
print(f"Best parameters:")
for key, value in study_lgb.best_params.items():
    print(f"  {key}: {value}")

# Optimize XGBoost for Revenue
print("\n" + "=" * 80)
print("OPTIMIZING XGBOOST HYPERPARAMETERS (REVENUE)")
print("=" * 80)
print("Running 30 trials (this may take 5-10 minutes)...")

study_xgb = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=RANDOM_SEED)
)
study_xgb.optimize(objective_xgb_revenue, n_trials=30, show_progress_bar=True)

print(f"\nBest RMSE: {study_xgb.best_value:,.2f}")
print(f"Best parameters:")
for key, value in study_xgb.best_params.items():
    print(f"  {key}: {value}")

# Store best parameters for Revenue
best_lgb_params_revenue = study_lgb.best_params.copy()
best_lgb_params_revenue.update({
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'verbose': -1,
    'seed': RANDOM_SEED,
    'n_jobs': -1
})

best_xgb_params_revenue = study_xgb.best_params.copy()
best_xgb_params_revenue.update({
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'seed': RANDOM_SEED,
    'tree_method': 'hist',
    'verbosity': 0
})

# Use same parameters for COGS (they share similar patterns)
best_lgb_params_cogs = best_lgb_params_revenue.copy()
best_xgb_params_cogs = best_xgb_params_revenue.copy()

print("\n✓ Optimization complete! Parameters will be used for both Revenue and COGS models.")

## 7. Train Optimized Models (Revenue + COGS)

Train final LightGBM and XGBoost models using the Optuna best parameters for both targets.

In [ ]:
# ============================================================================
# TRAIN OPTIMIZED MODELS FOR REVENUE AND COGS
# ============================================================================

print("=" * 80)
print("TRAINING FINAL MODELS WITH OPTIMIZED HYPERPARAMETERS")
print("=" * 80)

# -------------------------
# REVENUE MODELS
# -------------------------
print("\n[REVENUE] Training optimized LightGBM...")

lgb_train_final = lgb.Dataset(X_train, label=y_train_revenue)
lgb_val_final = lgb.Dataset(X_val, label=y_val_revenue, reference=lgb_train_final)

lgb_model_final = lgb.train(
    best_lgb_params_revenue,
    lgb_train_final,
    num_boost_round=1000,
    valid_sets=[lgb_val_final],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

lgb_pred_val_final = lgb_model_final.predict(X_val, num_iteration=lgb_model_final.best_iteration)
lgb_pred_test_final = lgb_model_final.predict(X_test, num_iteration=lgb_model_final.best_iteration)
lgb_metrics_val_revenue = calculate_metrics(y_val_revenue, lgb_pred_val_final, "LightGBM Revenue")

print("\n[REVENUE] Training optimized XGBoost...")

dtrain_final = xgb.DMatrix(X_train, label=y_train_revenue)
dval_final = xgb.DMatrix(X_val, label=y_val_revenue)
dtest_final = xgb.DMatrix(X_test)

xgb_model_final = xgb.train(
    best_xgb_params_revenue,
    dtrain_final,
    num_boost_round=1000,
    evals=[(dval_final, 'valid')],
    early_stopping_rounds=50,
    verbose_eval=100
)

xgb_pred_val_final = xgb_model_final.predict(dval_final, iteration_range=(0, xgb_model_final.best_iteration + 1))
xgb_pred_test_final = xgb_model_final.predict(dtest_final, iteration_range=(0, xgb_model_final.best_iteration + 1))
xgb_metrics_val_revenue = calculate_metrics(y_val_revenue, xgb_pred_val_final, "XGBoost Revenue")

# -------------------------
# COGS MODELS
# -------------------------
print("\n[COGS] Training optimized LightGBM...")

lgb_train_cogs = lgb.Dataset(X_train, label=y_train_cogs)
lgb_val_cogs = lgb.Dataset(X_val, label=y_val_cogs, reference=lgb_train_cogs)

lgb_model_cogs_final = lgb.train(
    best_lgb_params_cogs,
    lgb_train_cogs,
    num_boost_round=1000,
    valid_sets=[lgb_val_cogs],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

lgb_pred_val_cogs = lgb_model_cogs_final.predict(X_val, num_iteration=lgb_model_cogs_final.best_iteration)
lgb_pred_test_cogs = lgb_model_cogs_final.predict(X_test, num_iteration=lgb_model_cogs_final.best_iteration)
lgb_metrics_val_cogs = calculate_metrics(y_val_cogs, lgb_pred_val_cogs, "LightGBM COGS")

print("\n[COGS] Training optimized XGBoost...")

dtrain_cogs = xgb.DMatrix(X_train, label=y_train_cogs)
dval_cogs = xgb.DMatrix(X_val, label=y_val_cogs)
dtest_cogs = xgb.DMatrix(X_test)

xgb_model_cogs_final = xgb.train(
    best_xgb_params_cogs,
    dtrain_cogs,
    num_boost_round=1000,
    evals=[(dval_cogs, 'valid')],
    early_stopping_rounds=50,
    verbose_eval=100
)

xgb_pred_val_cogs = xgb_model_cogs_final.predict(dval_cogs, iteration_range=(0, xgb_model_cogs_final.best_iteration + 1))
xgb_pred_test_cogs = xgb_model_cogs_final.predict(dtest_cogs, iteration_range=(0, xgb_model_cogs_final.best_iteration + 1))
xgb_metrics_val_cogs = calculate_metrics(y_val_cogs, xgb_pred_val_cogs, "XGBoost COGS")

# Save models
MODEL_PATH.mkdir(exist_ok=True, parents=True)
lgb_model_final.save_model(str(MODEL_PATH / 'lightgbm_revenue_optimized.txt'))
xgb_model_final.save_model(str(MODEL_PATH / 'xgboost_revenue_optimized.json'))
lgb_model_cogs_final.save_model(str(MODEL_PATH / 'lightgbm_cogs_optimized.txt'))
xgb_model_cogs_final.save_model(str(MODEL_PATH / 'xgboost_cogs_optimized.json'))

print("\n✓ All optimized models trained and saved!")

## 8. Ensemble Weight Optimization

In [ ]:
# ============================================================================
# ENSEMBLE OPTIMIZATION FOR REVENUE AND COGS
# ============================================================================

print("=" * 80)
print("FINDING OPTIMAL ENSEMBLE WEIGHTS")
print("=" * 80)

# Test different weight combinations
weight_combinations = [(w/10, 1-w/10) for w in range(0, 11)]

# -------------------------
# REVENUE ENSEMBLE
# -------------------------
print("\nOptimizing Revenue ensemble...")
best_revenue_rmse = float('inf')
best_revenue_weights = None

for w_lgb, w_xgb in weight_combinations:
    ensemble_pred = w_lgb * lgb_pred_val_final + w_xgb * xgb_pred_val_final
    rmse = np.sqrt(mean_squared_error(y_val_revenue, ensemble_pred))
    
    if rmse < best_revenue_rmse:
        best_revenue_rmse = rmse
        best_revenue_weights = (w_lgb, w_xgb)

print(f"Best Revenue weights: LGB={best_revenue_weights[0]:.1f}, XGB={best_revenue_weights[1]:.1f}")
print(f"Best Revenue RMSE: {best_revenue_rmse:,.2f}")

# Final Revenue ensemble predictions
revenue_ensemble_test = (best_revenue_weights[0] * lgb_pred_test_final + 
                         best_revenue_weights[1] * xgb_pred_test_final)

# -------------------------
# COGS ENSEMBLE
# -------------------------
print("\nOptimizing COGS ensemble...")
best_cogs_rmse = float('inf')
best_cogs_weights = None

for w_lgb, w_xgb in weight_combinations:
    ensemble_pred = w_lgb * lgb_pred_val_cogs + w_xgb * xgb_pred_val_cogs
    rmse = np.sqrt(mean_squared_error(y_val_cogs, ensemble_pred))
    
    if rmse < best_cogs_rmse:
        best_cogs_rmse = rmse
        best_cogs_weights = (w_lgb, w_xgb)

print(f"Best COGS weights: LGB={best_cogs_weights[0]:.1f}, XGB={best_cogs_weights[1]:.1f}")
print(f"Best COGS RMSE: {best_cogs_rmse:,.2f}")

# Final COGS ensemble predictions
cogs_ensemble_test = (best_cogs_weights[0] * lgb_pred_test_cogs + 
                      best_cogs_weights[1] * xgb_pred_test_cogs)

print("\n✓ Optimal ensemble weights found!")

## 9. Generate Final Submission (Revenue + COGS)

In [ ]:
# ============================================================================
# GENERATE FINAL SUBMISSION FILE (Revenue + COGS)
# ============================================================================

print("=" * 80)
print("GENERATING FINAL SUBMISSION FILE")
print("=" * 80)

# Get test dates from test_df
test_dates = test_df['date'].values

# Create submission DataFrame with both Revenue and COGS
final_submission = pd.DataFrame({
    'Date': test_dates,
    'Revenue': revenue_ensemble_test,
    'COGS': cogs_ensemble_test
})

# Ensure Date is datetime
final_submission['Date'] = pd.to_datetime(final_submission['Date'])

# Sort by date
final_submission = final_submission.sort_values('Date').reset_index(drop=True)

# Verify dimensions
print(f"\nFinal submission shape: {final_submission.shape}")
print(f"Date range: {final_submission['Date'].min()} to {final_submission['Date'].max()}")
print(f"Total predictions: {len(final_submission)}")

# Display statistics
print("\n" + "=" * 80)
print("SUBMISSION STATISTICS")
print("=" * 80)

print("\nRevenue predictions:")
print(f"  Min: ${final_submission['Revenue'].min():,.2f}")
print(f"  Max: ${final_submission['Revenue'].max():,.2f}")
print(f"  Mean: ${final_submission['Revenue'].mean():,.2f}")
print(f"  Median: ${final_submission['Revenue'].median():,.2f}")

print("\nCOGS predictions:")
print(f"  Min: ${final_submission['COGS'].min():,.2f}")
print(f"  Max: ${final_submission['COGS'].max():,.2f}")
print(f"  Mean: ${final_submission['COGS'].mean():,.2f}")
print(f"  Median: ${final_submission['COGS'].median():,.2f}")

# Calculate predicted gross profit margin
predicted_margin = (final_submission['Revenue'] - final_submission['COGS']) / final_submission['Revenue']
print(f"\nPredicted Gross Profit Margin:")
print(f"  Mean: {predicted_margin.mean():.2%}")
print(f"  Median: {predicted_margin.median():.2%}")

# Display first and last few rows
print("\n" + "=" * 80)
print("SUBMISSION PREVIEW")
print("=" * 80)
print("\nFirst 10 rows:")
print(final_submission.head(10).to_string(index=False))

print("\nLast 10 rows:")
print(final_submission.tail(10).to_string(index=False))

# Save submission file
OUTPUT_PATH.mkdir(exist_ok=True, parents=True)
submission_path = OUTPUT_PATH / 'sales_submission.csv'
final_submission.to_csv(submission_path, index=False)

print("\n" + "=" * 80)
print("✓ SUBMISSION FILE SAVED")
print("=" * 80)
print(f"\nFile: {submission_path}")
print(f"Columns: {list(final_submission.columns)}")
print(f"Rows: {len(final_submission)}")
print(f"Size: {submission_path.stat().st_size / 1024:.2f} KB")

# Verify against sample submission
try:
    sample_submission = pd.read_csv(RAW_PATH / 'sample_submission.csv')
    sample_submission['Date'] = pd.to_datetime(sample_submission['Date'])
    
    # Check shape
    if final_submission.shape == sample_submission.shape:
        print("\n✓ Shape matches sample_submission.csv")
    else:
        print(f"\n⚠ Shape mismatch: {final_submission.shape} vs {sample_submission.shape}")
    
    # Check columns
    if list(final_submission.columns) == list(sample_submission.columns):
        print("✓ Columns match sample_submission.csv")
    else:
        print(f"⚠ Column mismatch: {list(final_submission.columns)} vs {list(sample_submission.columns)}")
    
    # Check dates
    if set(final_submission['Date']) == set(sample_submission['Date']):
        print("✓ All dates match sample_submission.csv")
    else:
        missing = set(sample_submission['Date']) - set(final_submission['Date'])
        extra = set(final_submission['Date']) - set(sample_submission['Date'])
        if missing:
            print(f"⚠ Missing dates: {len(missing)}")
        if extra:
            print(f"⚠ Extra dates: {len(extra)}")
        
except Exception as e:
    print(f"\n⚠ Could not verify against sample_submission.csv: {e}")

print("\n" + "=" * 80)
print("🎉 TRAINING PIPELINE COMPLETE!")
print("=" * 80)
print(f"\nFinal submission ready for Kaggle:")
print(f"  {submission_path}")
print(f"\nModels saved:")
print(f"  LightGBM Revenue: {MODEL_PATH / 'lightgbm_revenue_optimized.txt'}")
print(f"  XGBoost Revenue: {MODEL_PATH / 'xgboost_revenue_optimized.json'}")
print(f"  LightGBM COGS: {MODEL_PATH / 'lightgbm_cogs_optimized.txt'}")
print(f"  XGBoost COGS: {MODEL_PATH / 'xgboost_cogs_optimized.json'}")

## 10. Recursive Forecast Pipeline (Advanced)

Auto-regressive recursive strategy: models retrained on log-target with seasonal blend and bias correction. Does not require exogenous features, works purely from the target history.

In [ ]:
# ============================================================================
# LEADERBOARD OPTIMIZATION: RECURSIVE FORECAST WITHOUT FUTURE EXOGENOUS SHIFT
# ============================================================================

from itertools import product

print("=" * 80)
print("LEADERBOARD OPTIMIZATION - RECURSIVE STRATEGY")
print("=" * 80)

AR_LAGS = [1, 7, 14, 28, 56, 91, 182, 364]
AR_WINDOWS = [7, 14, 28, 56]


def make_calendar_row(ts):
    """Calendar features known in advance for any forecast date."""
    day_of_week = ts.dayofweek
    day_of_year = ts.dayofyear
    return {
        'year': ts.year,
        'month': ts.month,
        'day': ts.day,
        'day_of_week': day_of_week,
        'day_of_year': day_of_year,
        'week_of_year': int(ts.isocalendar().week),
        'quarter': ts.quarter,
        'is_weekend': int(day_of_week >= 5),
        'is_month_start': int(ts.is_month_start),
        'is_month_end': int(ts.is_month_end),
        'month_sin': np.sin(2 * np.pi * ts.month / 12),
        'month_cos': np.cos(2 * np.pi * ts.month / 12),
        'dow_sin': np.sin(2 * np.pi * day_of_week / 7),
        'dow_cos': np.cos(2 * np.pi * day_of_week / 7),
        'doy_sin': np.sin(2 * np.pi * day_of_year / 365),
        'doy_cos': np.cos(2 * np.pi * day_of_year / 365),
    }


def make_autoreg_row(history_series, ts):
    """Build one feature row from past target values only (no future leakage)."""
    row = make_calendar_row(ts)

    for lag in AR_LAGS:
        lag_date = ts - pd.Timedelta(days=lag)
        row[f'lag_{lag}'] = history_series.get(lag_date, np.nan)

    prev_hist = history_series.loc[: ts - pd.Timedelta(days=1)]
    for w in AR_WINDOWS:
        tail = prev_hist.tail(w)
        row[f'roll_mean_{w}'] = tail.mean() if len(tail) else np.nan
        row[f'roll_std_{w}'] = tail.std() if len(tail) else np.nan
        row[f'roll_min_{w}'] = tail.min() if len(tail) else np.nan
        row[f'roll_max_{w}'] = tail.max() if len(tail) else np.nan

    row['ewm_7'] = prev_hist.ewm(span=7, adjust=False).mean().iloc[-1] if len(prev_hist) else np.nan
    row['ewm_28'] = prev_hist.ewm(span=28, adjust=False).mean().iloc[-1] if len(prev_hist) else np.nan

    if pd.notna(row['lag_1']) and pd.notna(row['lag_7']):
        row['diff_1_7'] = row['lag_1'] - row['lag_7']
    else:
        row['diff_1_7'] = np.nan

    if pd.notna(row['lag_7']) and pd.notna(row['lag_14']):
        row['diff_7_14'] = row['lag_7'] - row['lag_14']
    else:
        row['diff_7_14'] = np.nan

    return row


def build_supervised_autoreg(series, end_date, required_lags=(1, 7, 28, 364)):
    """Build trainable matrix from historical target series."""
    series = series.loc[:end_date].sort_index().astype(float)

    rows = []
    targets = []
    idx = []

    for ts in series.index:
        hist_until_prev = series.loc[: ts - pd.Timedelta(days=1)]
        row = make_autoreg_row(hist_until_prev, ts)

        if any(pd.isna(row[f'lag_{lag}']) for lag in required_lags):
            continue

        rows.append(row)
        targets.append(series.loc[ts])
        idx.append(ts)

    X = pd.DataFrame(rows, index=idx)
    y = pd.Series(targets, index=idx)

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X, y


def recursive_forecast(lgb_model, xgb_model, feature_cols, history_series, forecast_dates, w_lgb, seasonal_alpha, bias):
    """Recursive multi-step forecast for future horizon."""
    history_series = history_series.copy().sort_index().astype(float)
    preds = []

    for ts in forecast_dates:
        row = make_autoreg_row(history_series, ts)
        X_row = pd.DataFrame([row]).reindex(columns=feature_cols)
        X_row = X_row.replace([np.inf, -np.inf], np.nan).fillna(0.0)

        pred_lgb = np.expm1(lgb_model.predict(X_row)[0])
        pred_xgb = np.expm1(xgb_model.predict(X_row)[0])

        model_pred = w_lgb * pred_lgb + (1 - w_lgb) * pred_xgb

        seasonal_ref = history_series.get(ts - pd.Timedelta(days=364), np.nan)
        if pd.isna(seasonal_ref):
            seasonal_ref = history_series.tail(28).mean()

        pred = seasonal_alpha * model_pred + (1 - seasonal_alpha) * seasonal_ref + bias
        pred = max(float(pred), 0.0)

        preds.append(pred)
        history_series.loc[ts] = pred

    return np.array(preds)


def train_recursive_target(target_col):
    print(f"\n{'-' * 80}")
    print(f"Training recursive pipeline for {target_col}")
    print(f"{'-' * 80}")

    target_series = (
        sales_df[['date', target_col]]
        .dropna()
        .sort_values('date')
        .set_index('date')[target_col]
        .astype(float)
    )

    train_end = pd.to_datetime(TRAIN_END)
    val_start = pd.to_datetime(VAL_START)
    val_end = pd.to_datetime(VAL_END)

    X_all, y_all = build_supervised_autoreg(target_series, end_date=val_end)

    X_train = X_all.loc[:train_end].copy()
    y_train = y_all.loc[:train_end].copy()

    y_train_log = np.log1p(y_train.clip(lower=0))

    # Balanced model setup to reduce overfit and keep runtime stable on Kaggle.
    lgb_model = lgb.LGBMRegressor(
        n_estimators=900,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.05,
        reg_lambda=0.2,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        objective='regression',
    )

    xgb_model = xgb.XGBRegressor(
        n_estimators=900,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=2,
        reg_alpha=0.05,
        reg_lambda=1.0,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method='hist',
        objective='reg:squarederror',
    )

    lgb_model.fit(X_train, y_train_log)
    xgb_model.fit(X_train, y_train_log)

    feature_cols = list(X_train.columns)

    val_dates = pd.date_range(start=val_start, end=val_end, freq='D')
    y_val_actual = target_series.loc[val_start:val_end].values

    best_rmse = np.inf
    best_config = None

    for w_lgb, seasonal_alpha in product([0.2, 0.35, 0.5, 0.65, 0.8], [0.6, 0.75, 0.85, 0.95, 1.0]):
        y_val_pred = recursive_forecast(
            lgb_model=lgb_model,
            xgb_model=xgb_model,
            feature_cols=feature_cols,
            history_series=target_series.loc[:train_end],
            forecast_dates=val_dates,
            w_lgb=w_lgb,
            seasonal_alpha=seasonal_alpha,
            bias=0.0,
        )

        rmse = np.sqrt(mean_squared_error(y_val_actual, y_val_pred))
        if rmse < best_rmse:
            best_rmse = rmse
            best_config = {
                'w_lgb': w_lgb,
                'seasonal_alpha': seasonal_alpha,
                'y_val_pred': y_val_pred,
            }

    bias = float(np.median(y_val_actual - best_config['y_val_pred']))
    y_val_pred_bias = best_config['y_val_pred'] + bias
    rmse_bias = np.sqrt(mean_squared_error(y_val_actual, y_val_pred_bias))
    mae_bias = mean_absolute_error(y_val_actual, y_val_pred_bias)

    print(f"Best config: w_lgb={best_config['w_lgb']:.2f}, seasonal_alpha={best_config['seasonal_alpha']:.2f}")
    print(f"Validation RMSE (no bias): {best_rmse:,.2f}")
    print(f"Validation RMSE (with bias): {rmse_bias:,.2f}")
    print(f"Validation MAE (with bias): {mae_bias:,.2f}")
    print(f"Bias correction: {bias:,.2f}")

    # Refit on all known data up to validation end for final test forecast.
    X_hist = X_all.loc[:val_end].copy()
    y_hist = y_all.loc[:val_end].copy()
    y_hist_log = np.log1p(y_hist.clip(lower=0))

    lgb_final = lgb.LGBMRegressor(
        n_estimators=900,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.05,
        reg_lambda=0.2,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        objective='regression',
    )

    xgb_final = xgb.XGBRegressor(
        n_estimators=900,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=2,
        reg_alpha=0.05,
        reg_lambda=1.0,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method='hist',
        objective='reg:squarederror',
    )

    lgb_final.fit(X_hist, y_hist_log)
    xgb_final.fit(X_hist, y_hist_log)

    test_dates = pd.date_range(start=pd.to_datetime(TEST_START), end=pd.to_datetime(TEST_END), freq='D')
    y_test_pred = recursive_forecast(
        lgb_model=lgb_final,
        xgb_model=xgb_final,
        feature_cols=feature_cols,
        history_series=target_series.loc[:val_end],
        forecast_dates=test_dates,
        w_lgb=best_config['w_lgb'],
        seasonal_alpha=best_config['seasonal_alpha'],
        bias=bias,
    )

    return {
        'target_col': target_col,
        'val_rmse': rmse_bias,
        'val_mae': mae_bias,
        'blend_w_lgb': best_config['w_lgb'],
        'blend_seasonal_alpha': best_config['seasonal_alpha'],
        'bias': bias,
        'test_pred': y_test_pred,
        
    }


# Train recursive pipelines for both targets
revenue_recursive = train_recursive_target('Revenue')
cogs_recursive = train_recursive_target('COGS')

# Build optimized submission
test_dates = pd.date_range(start=pd.to_datetime(TEST_START), end=pd.to_datetime(TEST_END), freq='D')
optimized_submission = pd.DataFrame({
    'Date': test_dates,
    'Revenue': np.maximum(revenue_recursive['test_pred'], 0.0),
    'COGS': np.maximum(cogs_recursive['test_pred'], 0.0),
})

# Keep margin in realistic historical band to avoid unstable tails.
hist_margin = (sales_df['Revenue'] - sales_df['COGS']) / sales_df['Revenue'].replace(0, np.nan)
margin_low = float(np.nanpercentile(hist_margin, 1))
margin_high = float(np.nanpercentile(hist_margin, 99))

safe_revenue = optimized_submission['Revenue'].replace(0, np.nan)
pred_margin = (optimized_submission['Revenue'] - optimized_submission['COGS']) / safe_revenue

mask_low = pred_margin < margin_low
mask_high = pred_margin > margin_high

optimized_submission.loc[mask_low, 'COGS'] = optimized_submission.loc[mask_low, 'Revenue'] * (1 - margin_low)
optimized_submission.loc[mask_high, 'COGS'] = optimized_submission.loc[mask_high, 'Revenue'] * (1 - margin_high)
optimized_submission['COGS'] = np.minimum(optimized_submission['COGS'], optimized_submission['Revenue'] * 0.999)
optimized_submission['COGS'] = np.maximum(optimized_submission['COGS'], 0.0)

# Save both versioned file and official file name.
optimized_submission_path = OUTPUT_PATH / 'sales_submission_recursive_v2.csv'
official_submission_path = OUTPUT_PATH / 'sales_submission.csv'

optimized_submission.to_csv(optimized_submission_path, index=False)
optimized_submission.to_csv(official_submission_path, index=False)

print("\n" + "=" * 80)
print("RECURSIVE OPTIMIZATION SUMMARY")
print("=" * 80)
print(f"Revenue val RMSE: {revenue_recursive['val_rmse']:,.2f}")
print(f"COGS val RMSE: {cogs_recursive['val_rmse']:,.2f}")
print(f"Average val RMSE: {(revenue_recursive['val_rmse'] + cogs_recursive['val_rmse']) / 2:,.2f}")
print(f"\nRevenue blend: w_lgb={revenue_recursive['blend_w_lgb']:.2f}, seasonal_alpha={revenue_recursive['blend_seasonal_alpha']:.2f}")
print(f"COGS blend: w_lgb={cogs_recursive['blend_w_lgb']:.2f}, seasonal_alpha={cogs_recursive['blend_seasonal_alpha']:.2f}")
print(f"\nSaved optimized file: {optimized_submission_path}")
print(f"Saved official file: {official_submission_path}")
print(f"Submission shape: {optimized_submission.shape}")
print(f"Revenue range: {optimized_submission['Revenue'].min():,.2f} -> {optimized_submission['Revenue'].max():,.2f}")
print(f"COGS range: {optimized_submission['COGS'].min():,.2f} -> {optimized_submission['COGS'].max():,.2f}")
